Objective:
To build a Recurrent Neural Network (RNN) model that classifies news articles into different categories using the AG News dataset. The assignment involves preprocessing text data, converting words into numerical sequences, training an RNN model, and evaluating its classification accuracy.

In [4]:
# ==========================================
# Step 1: Install and Load the Dataset
# ==========================================

# Install the datasets library (Run once)
!pip install datasets

# Import required libraries
from datasets import load_dataset
import pandas as pd

# Load AG News dataset
dataset = load_dataset("wangrongsheng/ag_news")

# Convert into DataFrame
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# Display first few rows
print(train_df.head())

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [6]:
# Step 2: Clean and Normalize the Text
import re

train_df["text"] = train_df["text"].str.lower()
train_df["text"] = train_df["text"].apply(lambda x: re.sub(r'[^a-zA-Z ]', '', x))



In [7]:
# Step 3: Tokenize the Text
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(train_df["text"])

X_train = tokenizer.texts_to_sequences(train_df["text"])
X_test = tokenizer.texts_to_sequences(test_df["text"])

In [8]:
# Step 4: Pad the Sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train = pad_sequences(X_train, maxlen=50)
X_test = pad_sequences(X_test, maxlen=50)


In [9]:
# Step 5: Convert Labels to Categorical
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(train_df["label"])
y_test = to_categorical(test_df["label"])



In [11]:
# Step 6: Build the RNN Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=64, input_length=50))
model.add(SimpleRNN(64))
model.add(Dense(4, activation="softmax"))


In [12]:
# Step 7: Compile and Train
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

model.fit(X_train, y_train,
          epochs=5,
          validation_split=0.2)


Epoch 1/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 67s 22ms/step - accuracy: 0.8317 - loss: 0.4770 - val_accuracy: 0.8859 - val_loss: 0.3591
Epoch 2/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 81s 21ms/step - accuracy: 0.9036 - loss: 0.3135 - val_accuracy: 0.8657 - val_loss: 0.4133
Epoch 3/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 62s 21ms/step - accuracy: 0.9190 - loss: 0.2610 - val_accuracy: 0.8878 - val_loss: 0.3674
Epoch 4/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 83s 21ms/step - accuracy: 0.9313 - loss: 0.2141 - val_accuracy: 0.8660 - val_loss: 0.4188
Epoch 5/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 81s 21ms/step - accuracy: 0.9392 - loss: 0.1876 - val_accuracy: 0.8831 - val_loss: 0.4468


In [14]:
# Step 8: Evaluate the Model

loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8911 - loss: 0.4340
Accuracy: 0.8910526037216187
